# Building Vector DB

Run this file to build Vector DB. 

# Before running 
In terminal, run the following code: 
```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
ollama serve
OLLAMA_NUM_GPU=0 ollama serve
ollama pull llama3.1:8b
```

In [3]:
# LOCAL CHANGE: No Google Colab API key setup needed
# We are using Ollama locally, so no OPENAI_API_KEY is required.

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

False

In [4]:
from pathlib import Path
from typing import List
import json
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import ChatOllama

# Load documents as PDFs
from langchain_community.document_loaders import PyPDFLoader

# Chunk documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create vector DB
from langchain_community.vectorstores import FAISS

print("Imports loaded.")


Imports loaded.


# Upload relevant files
Skip if already done. 

In [5]:
pdf_folder = Path("pdfs")
pdf_folder.mkdir(exist_ok=True)

pdf_files = sorted(pdf_folder.glob("*.pdf"))

print(f"PDF folder: {pdf_folder.resolve()}")
print(f"PDFs found: {len(pdf_files)}")

for pdf in pdf_files:
    print("-", pdf.name)

if not pdf_files:
    raise FileNotFoundError(
        "No PDFs found. Put your PDF files inside the local 'pdfs/' folder, then rerun this cell."
    )


PDF folder: /Users/julie/Documents/github/ds593_finalproject/pdfs
PDFs found: 45
- 3511265.3550437.pdf
- 3511265.3550438.pdf
- 3511265.3550439.pdf
- 3511265.3550440.pdf
- 3511265.3550441.pdf
- 3511265.3550442.pdf
- 3511265.3550443.pdf
- 3511265.3550444.pdf
- 3511265.3550445.pdf
- 3511265.3550446.pdf
- 3511265.3550447.pdf
- 3511265.3550448.pdf
- 3511265.3550449.pdf
- 3511265.3550450.pdf
- 3511265.3550451.pdf
- 3511265.3550452.pdf
- 3511265.3550497.pdf
- 3614407.3643696.pdf
- 3614407.3643697.pdf
- 3614407.3643698.pdf
- 3614407.3643699.pdf
- 3614407.3643700.pdf
- 3614407.3643701.pdf
- 3614407.3643702.pdf
- 3614407.3643703.pdf
- 3614407.3643704.pdf
- 3614407.3643705.pdf
- 3614407.3643706.pdf
- 3614407.3643707.pdf
- 3614407.3643708.pdf
- 3709025.3712206.pdf
- 3709025.3712207.pdf
- 3709025.3712208.pdf
- 3709025.3712209.pdf
- 3709025.3712210.pdf
- 3709025.3712211.pdf
- 3709025.3712212.pdf
- 3709025.3712213.pdf
- 3709025.3712214.pdf
- 3709025.3712215.pdf
- 3709025.3712216.pdf
- 3709025.3712217

In [7]:
# Agentic IEEE citation extraction from PDF tex

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def call_citation_llm(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def safe_json_parse(text: str) -> dict:
    """
    Ollama sometimes returns extra text around JSON.
    This safely extracts the JSON object.
    """
    text = text.strip()

    text = re.sub(r"^```json", "", text)
    text = re.sub(r"^```", "", text)
    text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())

    raise ValueError(f"No valid JSON found in LLM response:\n{text}")


def extract_pdf_front_matter(pdf_path: str, max_pages: int = 1) -> str:
    """
    Reads the first few pages of a PDF.
    Citation information is usually located there.
    """
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    front_pages = pages[:max_pages]

    return "\n\n".join(
        f"--- Page {i + 1} ---\n{page.page_content}"
        for i, page in enumerate(front_pages)
    )


def citation_agent(pdf_path: str) -> dict:
    """
    Agent that reads the PDF text and generates a full IEEE citation.
    """
    front_matter = extract_pdf_front_matter(pdf_path)
    front_matter = front_matter[:6000]

    prompt = f"""
You are a citation extraction agent.

Your task is to read the PDF text and create a full IEEE-style citation.

Return ONLY valid JSON in this exact format:

{{
  "title": "",
  "authors": "",
  "venue": "",
  "year": "",
  "publisher": "",
  "doi": "",
  "ieee_citation": ""
}}

Rules:
- Use only information visible in the PDF text.
- Do not invent missing fields.
- If a field is missing, write "Unknown".
- The ieee_citation must be a complete IEEE-style reference.
- Use this general IEEE format:
  Authors, "Title," Venue, Publisher, Year, doi: DOI.
- If there is no DOI, omit the DOI from the ieee_citation.
- Return JSON only.

PDF text:
{front_matter}
"""

    raw = call_citation_llm(prompt)

    return safe_json_parse(raw)


print("Citation agent ready.")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 7068.54it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Citation agent ready.


# Load documents as PDFs
Skip if already done

In [15]:
# Load PDFs and attach IEEE citations using DOI-first lookup

import requests
import re
from langchain_community.document_loaders import PyPDFLoader

docs = []
pdf_citations = {}

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def build_acm_doi_from_filename(pdf_path):
    """
    Example:
    3511265.3550440.pdf -> 10.1145/3511265.3550440
    """
    stem = pdf_path.stem

    if re.match(r"^\d+\.\d+$", stem):
        return f"10.1145/{stem}"

    return None


def lookup_citation_by_doi(doi):
    url = f"https://api.crossref.org/works/{doi}"

    response = requests.get(url, timeout=20)
    response.raise_for_status()

    item = response.json()["message"]

    title = item.get("title", ["Unknown"])[0]

    authors = []
    for author in item.get("author", []):
        given = author.get("given", "")
        family = author.get("family", "")
        name = clean_text(f"{given} {family}")
        if name:
            authors.append(name)

    authors_text = ", ".join(authors) if authors else "Unknown"

    venue_list = item.get("container-title", ["Unknown"])
    venue = venue_list[0] if venue_list else "Unknown"

    publisher = item.get("publisher", "Unknown")

    year = "Unknown"
    if "issued" in item:
        year = item["issued"]["date-parts"][0][0]

    doi = item.get("DOI", doi)

    ieee_citation = (
        f'{authors_text}, "{title}," {venue}, '
        f'{publisher}, {year}, doi: {doi}.'
    )

    return {
        "title": title,
        "authors": authors_text,
        "venue": venue,
        "year": year,
        "publisher": publisher,
        "doi": doi,
        "ieee_citation": ieee_citation
    }


def fallback_lookup_from_first_page(pdf_path):
    """
    Backup only if DOI lookup fails.
    Uses the first 500 characters, not the full page.
    """
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    first_page_text = clean_text(pages[0].page_content)

    url = "https://api.crossref.org/works"
    params = {
        "query.bibliographic": first_page_text[:500],
        "rows": 1
    }

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()

    item = response.json()["message"]["items"][0]

    title = item.get("title", ["Unknown"])[0]

    authors = []
    for author in item.get("author", []):
        given = author.get("given", "")
        family = author.get("family", "")
        name = clean_text(f"{given} {family}")
        if name:
            authors.append(name)

    authors_text = ", ".join(authors) if authors else "Unknown"

    venue_list = item.get("container-title", ["Unknown"])
    venue = venue_list[0] if venue_list else "Unknown"

    publisher = item.get("publisher", "Unknown")

    year = "Unknown"
    if "issued" in item:
        year = item["issued"]["date-parts"][0][0]

    doi = item.get("DOI", "Unknown")

    ieee_citation = (
        f'{authors_text}, "{title}," {venue}, '
        f'{publisher}, {year}, doi: {doi}.'
    )

    return {
        "title": title,
        "authors": authors_text,
        "venue": venue,
        "year": year,
        "publisher": publisher,
        "doi": doi,
        "ieee_citation": ieee_citation
    }


for pdf_path in pdf_files:
    print(f"\nProcessing: {pdf_path.name}")

    try:
        doi = build_acm_doi_from_filename(pdf_path)

        if doi:
            print("Trying DOI:", doi)
            citation_data = lookup_citation_by_doi(doi)
        else:
            print("No DOI-like filename found. Using first-page fallback.")
            citation_data = fallback_lookup_from_first_page(pdf_path)

        print("Found citation:", citation_data["ieee_citation"])

    except Exception as e:
        print(f"Could not find citation for {pdf_path.name}: {e}")

        citation_data = {
            "title": pdf_path.stem,
            "authors": "Unknown",
            "venue": "Unknown",
            "year": "Unknown",
            "publisher": "Unknown",
            "doi": "Unknown",
            "ieee_citation": f'Unknown, "{pdf_path.stem}," Unknown, Unknown.'
        }

    pdf_citations[pdf_path.name] = citation_data

    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()

    for page in pages:
        page.metadata["source_file"] = pdf_path.name
        page.metadata["source_path"] = str(pdf_path)
        page.metadata["page"] = page.metadata.get("page", 0) + 1

        page.metadata["title"] = citation_data["title"]
        page.metadata["authors"] = citation_data["authors"]
        page.metadata["venue"] = citation_data["venue"]
        page.metadata["year"] = citation_data["year"]
        page.metadata["publisher"] = citation_data["publisher"]
        page.metadata["doi"] = citation_data["doi"]
        page.metadata["ieee_citation"] = citation_data["ieee_citation"]

    docs.extend(pages)


print("\nTotal loaded pages:", len(docs))
print("Total PDFs:", len(pdf_files))

print("\nExtracted IEEE citations:")
for file_name, citation in pdf_citations.items():
    print(f"\n{file_name}")
    print(citation["ieee_citation"])


Processing: 3511265.3550437.pdf
Trying DOI: 10.1145/3511265.3550437
Found citation: Aniket Kesari, "The Privacy-Fairness-Accuracy Frontier," Proceedings of the 2022 Symposium on Computer Science and Law, ACM, 2022, doi: 10.1145/3511265.3550437.

Processing: 3511265.3550438.pdf
Trying DOI: 10.1145/3511265.3550438
Found citation: Jason D. Hartline, Daniel W. Linna, Liren Shan, Alex Tang, "Algorithmic Learning Foundations for Common Law," Proceedings of the 2022 Symposium on Computer Science and Law, ACM, 2022, doi: 10.1145/3511265.3550438.

Processing: 3511265.3550439.pdf
Trying DOI: 10.1145/3511265.3550439
Found citation: Peter Henderson, Ben Chugg, Brandon Anderson, Daniel E. Ho, "Beyond Ads: Sequential Decision-Making Algorithms in Law and Public Policy," Proceedings of the 2022 Symposium on Computer Science and Law, ACM, 2022, doi: 10.1145/3511265.3550439.

Processing: 3511265.3550440.pdf
Trying DOI: 10.1145/3511265.3550440
Found citation: Moon Duchin, Douglas Spencer, "Blind Justic

# Chunk documents

In [ ]:
# Chunk documents using fixed size chunking
# from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

fixed_chunks = splitter.split_documents(docs)

print(f"Chunks created: {len(fixed_chunks)}")

Chunks created: 5243


In [ ]:
# Document structure aware chunking

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def document_structure_chunking(docs):
    chunks = []

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=[
            "\nAbstract",
            "\nIntroduction",
            "\nBackground",
            "\nMethods",
            "\nMethodology",
            "\nResults",
            "\nDiscussion",
            "\nConclusion",
            "\nReferences",
            "\n\n",
            "\n",
            ". ",
            " "
        ]
    )

    for doc in docs:
        split_docs = splitter.split_documents([doc])

        for chunk in split_docs:
            chunk.metadata["chunking_method"] = "document_structure"

        chunks.extend(split_docs)

    return chunks


structure_chunks = document_structure_chunking(docs)

print("Document structure chunks:", len(structure_chunks))

In [ ]:
# Sentence based chunking 

import re
from langchain_core.documents import Document

def sentence_chunking(docs, sentences_per_chunk=6, overlap_sentences=2):
    chunks = []

    for doc in docs:
        text = doc.page_content

        sentences = re.split(r'(?<=[.!?])\s+', text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 20]

        step = sentences_per_chunk - overlap_sentences

        for i in range(0, len(sentences), step):
            chunk_sentences = sentences[i:i + sentences_per_chunk]

            if not chunk_sentences:
                continue

            chunk_text = " ".join(chunk_sentences)

            metadata = doc.metadata.copy()
            metadata["chunking_method"] = "sentence"
            metadata["sentence_start"] = i
            metadata["sentence_end"] = i + len(chunk_sentences)

            chunks.append(Document(
                page_content=chunk_text,
                metadata=metadata
            ))

    return chunks


sentence_chunks = sentence_chunking(
    docs,
    sentences_per_chunk=6,
    overlap_sentences=2
)

print("Sentence chunks:", len(sentence_chunks))

In [ ]:
# Semantic chunking 

from langchain_experimental.text_splitter import SemanticChunker

def semantic_chunking(docs, embeddings):
    splitter = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=80
    )

    chunks = splitter.split_documents(docs)

    for chunk in chunks:
        chunk.metadata["chunking_method"] = "semantic"

    return chunks


semantic_chunks = semantic_chunking(docs, embeddings)

print("Semantic chunks:", len(semantic_chunks))

# Embedding Model

In [17]:
# Embedding Model. Using free tier AI
# from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8723.66it/s]


# Create Vector DB

In [ ]:
# Create vector DB
# from langchain_community.vectorstores import FAISS

fixed_vector_db = FAISS.from_documents(fixed_chunks, embeddings)
structure_vector_db = FAISS.from_documents(structure_chunks, embeddings)
sentence_vector_db = FAISS.from_documents(sentence_chunks, embeddings)
semantic_vector_db = FAISS.from_documents(semantic_chunks, embeddings)

print("All vector DBs created.")

Vector DB created!


In [ ]:
import shutil
from pathlib import Path

save_path = "fixed_rag_faiss_index"

# Remove existing index if it exists
if Path(save_path).exists():
    shutil.rmtree(save_path)

# Save fresh index
fixed_vector_db.save_local(save_path)

print(f"Vector DB saved locally to: {Path(save_path).resolve()}")

Vector DB saved locally to: /Users/julie/Documents/github/ds593_finalproject/fixed_rag_faiss_index
